# MEDUSA Reproduction Colab notebook

This notebook serves as a thin launcher. All logic is contained in the `.py` files inside the `code/` directory.

In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import drive
import sys

path = "/content/gdrive/MyDrive/CS4782/final_proj" # change based on where files are

drive.mount('/content/gdrive')

base_dir = path
sys.path.append(base_dir)
%cd $path

In [ ]:
# Full training run (paper-scale). Use --max_samples 1000 for a quick smoke test.
%run code/train.py --max_samples 60000 --save_path medusa_heads_tinyllama.pt

In [ ]:
%run code/benchmark.py --mode greedy

# Medusa Inference (Greedy Acceptance)

Run the full propose → verify → accept loop with greedy acceptance.  
Requires `results/medusa_heads_tinyllama.pt` from the training cell above.  
Prints per-prompt acceptance rate (extra tree tokens accepted per step) and TPS.

In [ ]:
%run code/benchmark.py --mode medusa --acceptance greedy --tree_budget 64 --checkpoint results/medusa_heads_tinyllama.pt

## Full TinyLlama Benchmark

Runs greedy baseline + Medusa inference + per-head accuracy in one pass.  
Saves `results/TinyLlama-1.1B-Chat-v1.0_benchmark.json` with speedup ratio and all metrics.

In [ ]:
%run code/benchmark.py --mode full --model_id TinyLlama/TinyLlama-1.1B-Chat-v1.0 --max_new_tokens 128 --checkpoint results/medusa_heads_tinyllama.pt

## TinyLlama — Typical Acceptance & Tree Budget Tuning

Compare greedy vs typical acceptance criterion (paper §2.3.1) and tune the tree budget.  
Runs below save to `results/comparison_tinyllama_budget64.json` and `results/comparison_tinyllama_budget32.json`.

In [ ]:
# Greedy vs typical acceptance — 64-node tree
%run code/benchmark.py --mode medusa --compare --tree_budget 64 --checkpoint results/medusa_heads_tinyllama.pt
!mv -f results/comparison.json results/comparison_tinyllama_budget64.json

In [ ]:
# Tree budget tuning — 32-node tree for speed vs acceptance tradeoff
%run code/benchmark.py --mode medusa --compare --tree_budget 32 --checkpoint results/medusa_heads_tinyllama.pt
!mv -f results/comparison.json results/comparison_tinyllama_budget32.json

## TinyLlama — Table 3 Ablation (heads-only / naive tree / optimized tree)

Reproduces paper Table 3 rows 1–3: speedup contribution of each technique.
- Row 1 (`tree=none`): heads-only linear chain, no tree attention — paper target ~1.54x.
- Row 2 (`tree=naive`): full 220-node Cartesian product — paper target ~1.92x.
- Row 3 (`tree=optimized`): 64-node pruned tree (MEDUSA-1 headline) — paper target ~2.18x.

In [ ]:
# Table 3 ablation — runs {none, naive, optimized} + shared greedy baseline.
# Saves results/table3_TinyLlama-1.1B-Chat-v1.0.json (consumed by visualize.py).
%run code/benchmark.py --mode table3 --model_id TinyLlama/TinyLlama-1.1B-Chat-v1.0 --max_new_tokens 128 --checkpoint results/medusa_heads_tinyllama.pt

## Vicuna-7B Scale Run

Paper-scale reproduction target. Backbone is loaded in 4-bit (bitsandbytes nf4) via `--quantize` so the model fits on a 24 GB GPU (L4 / A100 40GB).  
Trains fresh heads on Vicuna-7B and saves them to `results/medusa_heads_vicuna.pt` (kept separate from the TinyLlama checkpoint), then runs the full greedy + Medusa + head-accuracy benchmark.

In [ ]:
%run code/train.py --model_name lmsys/vicuna-7b-v1.5 --max_samples 60000 --save_path medusa_heads_vicuna.pt --quantize

In [ ]:
%run code/benchmark.py --mode full --model_id lmsys/vicuna-7b-v1.5 --max_new_tokens 128 --checkpoint results/medusa_heads_vicuna.pt --quantize

## Vicuna-7B — Typical Acceptance & Tree Budget Tuning

Same greedy-vs-typical + tree-budget sweep as TinyLlama, but on the paper-scale backbone.  
Runs below save to `results/comparison_vicuna_budget64.json` and `results/comparison_vicuna_budget32.json`.

In [ ]:
# Greedy vs typical acceptance — 64-node tree
%run code/benchmark.py --mode medusa --compare --tree_budget 64 --model_id lmsys/vicuna-7b-v1.5 --checkpoint results/medusa_heads_vicuna.pt --quantize
!mv -f results/comparison.json results/comparison_vicuna_budget64.json

In [ ]:
# Tree budget tuning — 32-node tree for speed vs acceptance tradeoff
%run code/benchmark.py --mode medusa --compare --tree_budget 32 --model_id lmsys/vicuna-7b-v1.5 --checkpoint results/medusa_heads_vicuna.pt --quantize
!mv -f results/comparison.json results/comparison_vicuna_budget32.json

## Vicuna-7B — Table 3 Ablation

Same three ablation rows as TinyLlama, but on the paper-scale backbone (4-bit). Headline speedup target for Row 3 is 2.18x (paper §3.1 / Table 3).

In [ ]:
# Table 3 ablation on Vicuna-7B (paper-scale).
# Saves results/table3_vicuna-7b-v1.5.json (consumed by visualize.py).
%run code/benchmark.py --mode table3 --model_id lmsys/vicuna-7b-v1.5 --max_new_tokens 128 --checkpoint results/medusa_heads_vicuna.pt --quantize

In [ ]:
%run code/visualize.py